# BO vs LHS Online Comparison

This notebook compares completed online BCLF campaigns for BO and LHS. It is analysis-only: it reads existing CSV artifacts, treats the scalar objective as a minimization target, and writes derived comparison tables and figures.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "experiments").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TARGET_TRIALS = 80
OUTPUT_DIR = REPO_ROOT / "experiments" / "bo_lhs_online_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_CONFIG = {
    "BO": {
        "root": REPO_ROOT / "experiments" / "bo_online_general",
        "stem": "bo_online_general",
        "color": "#1f77b4",
    },
    "LHS": {
        "root": REPO_ROOT / "experiments" / "lhs_online_general",
        "stem": "lhs_online_general",
        "color": "#2ca02c",
    },
}
METHOD_ORDER = pd.CategoricalDtype(categories=list(METHOD_CONFIG), ordered=True)

plt.style.use("seaborn-v0_8-whitegrid")

print(f"repo root: {REPO_ROOT}")
print(f"output: {OUTPUT_DIR}")

## Load Campaign Summaries and Histories

In [ ]:
def relative_to_repo(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


def finite_numeric(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.replace([np.inf, -np.inf], np.nan)


def as_bool(value: object) -> bool:
    if pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_campaign_summary(method: str, config: dict[str, object]) -> pd.DataFrame:
    path = Path(config["root"]) / "campaign_summary.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path).copy()
    df["method"] = method
    df["included"] = (df["status"] == "completed") & (df["num_trials"] >= TARGET_TRIALS)
    df["summary_path"] = relative_to_repo(path)
    return df


def load_history(method: str, config: dict[str, object], seed: int) -> pd.DataFrame:
    path = Path(config["root"]) / f"{config['stem']}_seed{seed}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path).copy()
    df["method"] = method
    df["seed"] = int(seed)
    df["history_path"] = relative_to_repo(path)
    df["iteration"] = pd.to_numeric(df["iteration"], errors="raise").astype(int)
    df["objective_numeric"] = finite_numeric(df["objective"])

    if "best_so_far" in df.columns:
        best = finite_numeric(df["best_so_far"])
        if best.isna().all():
            best = df["objective_numeric"].cummin()
    else:
        best = df["objective_numeric"].cummin()
    df["best_so_far"] = best.ffill()
    return df


summary_df = pd.concat(
    [load_campaign_summary(method, config) for method, config in METHOD_CONFIG.items()],
    ignore_index=True,
)
summary_df["method"] = summary_df["method"].astype(METHOD_ORDER)
summary_df = summary_df.sort_values(["method", "seed"]).reset_index(drop=True)

included_summary_df = summary_df.loc[summary_df["included"]].copy()
if included_summary_df.empty:
    raise ValueError("No completed BO/LHS seeds passed the inclusion gate.")

history_df = pd.concat(
    [
        load_history(str(row.method), METHOD_CONFIG[str(row.method)], int(row.seed))
        for row in included_summary_df.itertuples(index=False)
    ],
    ignore_index=True,
)
history_df["method"] = history_df["method"].astype(METHOD_ORDER)
history_df = history_df.sort_values(["method", "seed", "iteration"]).reset_index(drop=True)

display(
    summary_df[
        [
            "method",
            "seed",
            "status",
            "num_trials",
            "included",
            "best_objective",
            "cache_hit_ratio",
            "runtime_engine_sec",
            "runtime_sim_sec",
        ]
    ]
)

## Final Best Objective Tables

In [ ]:
diagnostic_rows = []
for (method, seed), group in history_df.groupby(["method", "seed"], observed=True):
    statuses = group["simulation_status"].fillna("missing")
    cache_hits = group["cache_hit"].map(as_bool) if "cache_hit" in group.columns else pd.Series(False, index=group.index)
    diagnostic_rows.append(
        {
            "method": method,
            "seed": int(seed),
            "history_rows": int(len(group)),
            "final_iteration": int(group["iteration"].max()),
            "done_rows": int((statuses == "done").sum()),
            "failed_rows": int((statuses == "failed").sum()),
            "finite_objective_rows": int(group["objective_numeric"].notna().sum()),
            "infinite_objective_rows": int(group["objective_numeric"].isna().sum()),
            "cache_hit_rows": int(cache_hits.sum()),
            "history_final_best": float(group["best_so_far"].iloc[-1]),
        }
    )

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df["method"] = diagnostic_df["method"].astype(METHOD_ORDER)
diagnostic_df = diagnostic_df.sort_values(["method", "seed"]).reset_index(drop=True)

final_by_method_seed_df = included_summary_df[
    [
        "method",
        "seed",
        "status",
        "num_trials",
        "best_objective",
        "first_feasible_iteration",
        "cache_hit_ratio",
        "runtime_engine_sec",
        "runtime_sim_sec",
    ]
].merge(diagnostic_df, on=["method", "seed"], how="left")
final_by_method_seed_df["summary_minus_history_final_best"] = (
    final_by_method_seed_df["best_objective"] - final_by_method_seed_df["history_final_best"]
)

final_summary_df = (
    final_by_method_seed_df.groupby("method", observed=True)
    .agg(
        n=("seed", "nunique"),
        median_best_objective=("best_objective", "median"),
        mean_best_objective=("best_objective", "mean"),
        best_min_objective=("best_objective", "min"),
        worst_max_objective=("best_objective", "max"),
        median_cache_hit_ratio=("cache_hit_ratio", "median"),
        median_runtime_engine_sec=("runtime_engine_sec", "median"),
        median_runtime_sim_sec=("runtime_sim_sec", "median"),
        total_failed_rows=("failed_rows", "sum"),
    )
    .reset_index()
)

paired_df = final_by_method_seed_df.pivot(index="seed", columns="method", values="best_objective").reset_index()
paired_df.columns.name = None
paired_df = paired_df.rename(columns={"BO": "bo_best_objective", "LHS": "lhs_best_objective"})
paired_df["lhs_improvement_over_bo"] = paired_df["bo_best_objective"] - paired_df["lhs_best_objective"]
paired_df["winner"] = np.select(
    [paired_df["lhs_improvement_over_bo"] > 0, paired_df["lhs_improvement_over_bo"] < 0],
    ["LHS", "BO"],
    default="Tie",
)

final_summary_path = OUTPUT_DIR / "bo_lhs_final_summary.csv"
paired_path = OUTPUT_DIR / "bo_lhs_per_seed_final.csv"
diagnostic_path = OUTPUT_DIR / "bo_lhs_status_runtime_diagnostics.csv"

final_summary_df.to_csv(final_summary_path, index=False)
paired_df.to_csv(paired_path, index=False)
final_by_method_seed_df.to_csv(diagnostic_path, index=False)

display(final_summary_df)
display(paired_df)
display(final_by_method_seed_df)

## Best-So-Far by Iteration

In [ ]:
best_so_far_by_iteration_df = (
    history_df.groupby(["method", "iteration"], observed=True)["best_so_far"]
    .agg(
        n="count",
        median="median",
        mean="mean",
        q25=lambda values: values.quantile(0.25),
        q75=lambda values: values.quantile(0.75),
        min="min",
        max="max",
    )
    .reset_index()
)
best_so_far_by_iteration_df["trial"] = best_so_far_by_iteration_df["iteration"] + 1

best_so_far_path = OUTPUT_DIR / "bo_lhs_best_so_far_by_iteration.csv"
best_so_far_by_iteration_df.to_csv(best_so_far_path, index=False)

display(best_so_far_by_iteration_df.head())
display(best_so_far_by_iteration_df.tail())

## Plots

In [ ]:
median_plot_path = OUTPUT_DIR / "bo_lhs_best_so_far_median.png"
final_plot_path = OUTPUT_DIR / "bo_lhs_final_best_by_seed.png"

fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=150)
for method, config in METHOD_CONFIG.items():
    color = config["color"]
    method_history = history_df.loc[history_df["method"] == method]
    for _, seed_history in method_history.groupby("seed"):
        seed_history = seed_history.sort_values("iteration")
        ax.plot(
            seed_history["iteration"] + 1,
            seed_history["best_so_far"],
            color=color,
            alpha=0.18,
            linewidth=1.0,
        )

    method_summary = best_so_far_by_iteration_df.loc[
        best_so_far_by_iteration_df["method"] == method
    ].sort_values("iteration")
    trial = method_summary["trial"].to_numpy(dtype=float)
    median = method_summary["median"].to_numpy(dtype=float)
    q25 = method_summary["q25"].to_numpy(dtype=float)
    q75 = method_summary["q75"].to_numpy(dtype=float)
    ax.fill_between(trial, q25, q75, color=color, alpha=0.14)
    ax.plot(trial, median, color=color, linewidth=2.4, label=f"{method} median")

ax.set_title("Online best-so-far objective")
ax.set_xlabel("Trial")
ax.set_ylabel("Best objective so far (lower is better)")
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(median_plot_path, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=150)
x = paired_df["seed"].to_numpy(dtype=float)
width = 0.36
ax.bar(x - width / 2, paired_df["bo_best_objective"], width, label="BO", color=METHOD_CONFIG["BO"]["color"])
ax.bar(x + width / 2, paired_df["lhs_best_objective"], width, label="LHS", color=METHOD_CONFIG["LHS"]["color"])
ax.set_title("Final best objective by seed")
ax.set_xlabel("Seed")
ax.set_ylabel("Final best objective (lower is better)")
ax.set_xticks(x)
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(final_plot_path, bbox_inches="tight")
plt.show()

## Output Checks

In [ ]:
required_outputs = [
    final_summary_path,
    paired_path,
    best_so_far_path,
    median_plot_path,
    final_plot_path,
]
missing_outputs = [path for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(missing_outputs)

expected_methods = set(METHOD_CONFIG)
actual_methods = set(final_summary_df["method"].astype(str))
if actual_methods != expected_methods:
    raise ValueError(f"Unexpected methods: {actual_methods}")

history_mismatch = final_by_method_seed_df.loc[
    final_by_method_seed_df["summary_minus_history_final_best"].abs() > 1e-9
]
if not history_mismatch.empty:
    raise ValueError("Summary and history final best values do not match.")

print("Wrote comparison artifacts:")
for path in required_outputs + [diagnostic_path]:
    print(f"- {relative_to_repo(path)}")